In [1]:
import os
import pandas as pd
import numpy as np
import random
import optuna
import json

# Sembunyikan log Optuna agar terminal bersih
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────────────────────────────
# PARAMETER GLOBAL & KONFIGURASI
# ─────────────────────────────────────────────────────────────────────
TARGET_FOLDER = os.path.join('osmnx_inputs')
MAX_KM_USER = 25.0

# ─────────────────────────────────────────────────────────────────────
# 1. FUNGSI FITNESS ACO (MULTI-WEEKEND & TERKALIBRASI)
# ─────────────────────────────────────────────────────────────────────
def hitung_fitness_aco_final(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user):
    w_jarak = 10.0
    w_jalan = 5.0
    w_penalti_cluster = 50.0 
    w_hari = 30.0  
    
    fitness_total = len(solusi_multi_hari) * w_hari
    log_itinerary = []
    
    for idx_hari, rute in enumerate(solusi_multi_hari, start=1):
        if idx_hari % 2 != 0:
            nama_hari = f"Pekan {(idx_hari+1)//2} - Sabtu (Jalan Berat)"
            w_penalti_fatigue = 1.5; w_penalti_jarak = 5.0     
        else:
            nama_hari = f"Pekan {idx_hari//2} - Minggu (Recovery)"
            w_penalti_fatigue = 8.0; w_penalti_jarak = 25.0    
            
        jarak_m = fatigue = skor_jalan = p_cluster = 0.0
        
        for k in range(len(rute) - 1):
            a, t = rute[k], rute[k+1]
            
            dm = dist_np[a, t]
            dz = elev_np[a, t]
            js = road_np[a, t]
            
            jarak_m += dm
            skor_jalan += js
            
            if dz > 0 and dm > 0: 
                fatigue += dm * ((dz / dm) ** 2) * 400.0
            else: 
                fatigue += dm * 0.0001
                
            if a != 0 and t != 0:
                if cluster_np[a-1] != cluster_np[t-1]:
                    p_cluster += w_penalti_cluster

        jkm = jarak_m / 1000.0
        p_jarak = max(0.0, jkm - max_km_user) * w_penalti_jarak  
        p_fatigue = fatigue * w_penalti_fatigue
            
        fitness_hari = (jkm * w_jarak) + (skor_jalan * w_jalan) + p_jarak + p_fatigue + p_cluster
        fitness_total += fitness_hari
        
        log_itinerary.append({
            "Hari/Trip": nama_hari, "Jarak (Km)": round(jkm, 2),
            "Fatigue Index": round(fatigue, 2), "Skor Jalan OSMnx": round(skor_jalan, 2),
            "P_Fatigue": round(p_fatigue, 2), "P_Cluster": round(p_cluster, 2)
        })
        
    return fitness_total, log_itinerary

# ─────────────────────────────────────────────────────────────────────
# 2. CORE ENGINE ACO
# ─────────────────────────────────────────────────────────────────────
def run_aco_final(dist_np, elev_np, road_np, cluster_np, alpha, beta, evaporation, max_km_user, num_ants=15, iterations=30):
    num_nodes = len(dist_np)
    max_m = max_km_user * 1000
    pheromone = np.full((num_nodes, num_nodes), 0.1)
    
    max_dist = dist_np.max()
    max_elev = elev_np.max() 
    if max_elev <= 0: max_elev = 1.0 
    
    best_score = float('inf')
    best_route = []
    best_log = None
    
    for _ in range(iterations):
        list_solusi_semut = []
        list_fitness_semut = []
        list_log_semut = []
        
        for ant in range(num_ants):
            destinasi_tersisa = list(range(1, num_nodes))
            rute_multi_hari = []
            
            while len(destinasi_tersisa) > 0:
                rute_hari_ini = [0]
                jarak_hari = 0.0
                
                while len(destinasi_tersisa) > 0:
                    curr = rute_hari_ini[-1]
                    
                    valid_candidates = []
                    for cand in destinasi_tersisa:
                        uji_dist = dist_np[curr, cand]
                        jarak_kembali_hotel = dist_np[cand, 0]
                        if jarak_hari + uji_dist + jarak_kembali_hotel <= max_m:
                            valid_candidates.append(cand)
                            
                    if not valid_candidates:
                        break 
                        
                    probs = []
                    for cand in valid_candidates:
                        tau = pheromone[curr][cand] ** alpha
                        d_ij = dist_np[curr, cand]
                        road_ij = road_np[curr, cand]
                        dz = elev_np[curr, cand]
                        
                        d_norm = d_ij / max_dist
                        r_norm = road_ij / 5.0
                        
                        cluster_norm = 0.0
                        if curr != 0 and cand != 0:
                            if cluster_np[curr-1] != cluster_np[0-1]: 
                                cluster_norm = 0.15 
                                
                        if len(rute_multi_hari) % 2 == 0:
                            eta = 1.0 / (0.8 * d_norm + 0.2 * r_norm + cluster_norm + 0.001)
                        else:
                            f_local = max(0, dz)
                            f_norm = f_local / (max_elev + 1e-6) 
                            eta = 1.0 / (0.5 * d_norm + 0.2 * r_norm + 0.3 * f_norm + cluster_norm + 0.001)
                            
                        probs.append(tau * (eta ** beta))
                        
                    sum_p = sum(probs)
                    if sum_p == 0: next_c = random.choice(valid_candidates)
                    else: next_c = random.choices(valid_candidates, weights=[p/sum_p for p in probs], k=1)[0]
                    
                    rute_hari_ini.append(next_c)
                    jarak_hari += dist_np[curr, next_c]
                    destinasi_tersisa.remove(next_c)
                    
                rute_hari_ini.append(0)
                
                if len(rute_hari_ini) == 2:
                    raise ValueError(f"Infeasible: Terdapat rute destinasi yang jarak pulang-perginya melebihi limit harian user ({MAX_KM_USER} km).")
                
                rute_multi_hari.append(rute_hari_ini)
                
            f_score, lg = hitung_fitness_aco_final(rute_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user)
            list_solusi_semut.append(rute_multi_hari)
            list_fitness_semut.append(f_score)
            list_log_semut.append(lg)
            
            if f_score < best_score:
                best_score = f_score
                best_route = rute_multi_hari
                best_log = lg
                
        pheromone *= (1.0 - evaporation)
        for idx, solusi in enumerate(list_solusi_semut):
            fit_value = list_fitness_semut[idx]
            if fit_value > 0:
                deposit = 2000 / fit_value
                for rute in solusi:
                    for k in range(len(rute) - 1):
                        pheromone[rute[k]][rute[k+1]] += deposit
                        
    return best_score, best_route, best_log

# ─────────────────────────────────────────────────────────────────────
# 3. GENERATE DASHBOARD HTML
# ─────────────────────────────────────────────────────────────────────
def generate_dashboard(rute_terbaik, log_metrik, names, route_registry, target_folder):
    print("\n" + "=" * 65)
    print("🌍 MEMBUAT DASHBOARD PETA INTERAKTIF...")
    
    OUTPUT_DASHBOARD = os.path.join(target_folder, 'dashboard_interaktif_aco.html')
    
    data_metrik = []
    for log in log_metrik:
        data_metrik.append({
            "hari": log["Hari/Trip"],
            "jarak": log["Jarak (Km)"],
            "fatigue": log["Fatigue Index"],
            "jalan": log["Skor Jalan OSMnx"]
        })
        
    data_rute_js = []
    for rute in rute_terbaik:
        hari_data = []
        for k in range(len(rute) - 1):
            idx_asal = rute[k]
            idx_tujuan = rute[k+1]
            key = f"{idx_asal}_{idx_tujuan}"
            
            if key in route_registry:
                jalur_detail = route_registry[key]
                hari_data.append({
                    "asal_nama": names[idx_asal],
                    "tujuan_nama": names[idx_tujuan],
                    "is_depot_asal": (idx_asal == 0),
                    "is_depot_tujuan": (idx_tujuan == 0),
                    "jalur": jalur_detail 
                })
        data_rute_js.append(hari_data)

    json_rute_str = json.dumps(data_rute_js)
    json_metrik_str = json.dumps(data_metrik)

    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Dashboard Optimasi Rute Sepeda ACO - Surabaya</title>
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
        
        <style>
            body {{ margin: 0; padding: 0; font-family: 'Inter', sans-serif; display: flex; height: 100vh; background-color: #f8f9fa; }}
            #map-container {{ flex: 1; height: 100%; position: relative; }}
            #map {{ height: 100%; width: 100%; }}
            .legend {{ position: absolute; bottom: 30px; left: 30px; z-index: 1000; background: white; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
            .legend-title {{ font-weight: bold; margin-bottom: 8px; font-size: 14px; }}
            .gradient-bar {{ width: 250px; height: 15px; border-radius: 4px; background: linear-gradient(to right, #27ae60, #f1c40f, #e74c3c); }}
            .legend-labels {{ display: flex; justify-content: space-between; margin-top: 5px; font-size: 12px; color: #555; }}
            #sidebar {{ width: 400px; height: 100%; background: white; box-shadow: -2px 0 10px rgba(0,0,0,0.1); overflow-y: auto; padding: 20px; box-sizing: border-box; z-index: 1000; }}
            h2 {{ margin-top: 0; color: #2c3e50; font-size: 20px; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
            .card {{ background: #ffffff; border: 1px solid #e0e6ed; border-radius: 8px; padding: 15px; margin-bottom: 15px; cursor: pointer; transition: all 0.3s ease; }}
            .card:hover {{ transform: translateY(-3px); box-shadow: 0 5px 15px rgba(0,0,0,0.08); border-color: #3498db; }}
            .card.active {{ border: 2px solid #3498db; background-color: #f0f7fb; }}
            .card-header {{ font-weight: 700; color: #2c3e50; margin-bottom: 10px; font-size: 15px; }}
            .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; color: white; margin-bottom: 10px; }}
            .badge.sabtu {{ background-color: #e74c3c; }}
            .badge.minggu {{ background-color: #27ae60; }}
            .metric-row {{ display: flex; justify-content: space-between; margin-bottom: 5px; font-size: 13px; color: #555; }}
            .metric-val {{ font-weight: bold; color: #2c3e50; }}
            #btn-semua {{ width: 100%; padding: 12px; background: #34495e; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; margin-bottom: 20px; transition: background 0.2s; }}
            #btn-semua:hover {{ background: #2c3e50; }}
        </style>
    </head>
    <body>
        <div id="map-container">
            <div id="map"></div>
            <div class="legend">
                <div class="legend-title">Elevasi Tanjakan Rute (m)</div>
                <div class="gradient-bar"></div>
                <div class="legend-labels">
                    <span>0m (Datar)</span>
                    <span>Landai</span>
                    <span>>25m (Berat)</span>
                </div>
            </div>
        </div>
        <div id="sidebar">
            <h2>Itinerary Multi-Hari ACO</h2>
            <button id="btn-semua" onclick="tampilkanSemuaRute()">Tampilkan Seluruh Rute</button>
            <div id="cards-container"></div>
        </div>
        <script>
            const dataRute = {json_rute_str};
            const dataMetrik = {json_metrik_str};
            
            const map = L.map('map').setView([-7.262015, 112.739727], 13);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
                attribution: '&copy; OpenStreetMap contributors &copy; CARTO'
            }}).addTo(map);

            let layerGroup = L.layerGroup().addTo(map);

            function getColorForElevation(elev) {{
                const minE = 0, midE = 12.5, maxE = 25;
                let val = Math.max(minE, Math.min(elev, maxE)); 
                let r, g, b;
                if (val <= midE) {{
                    let ratio = val / midE;
                    r = Math.round(0x27 + ratio * (0xf1 - 0x27));
                    g = Math.round(0xae + ratio * (0xc4 - 0xae));
                    b = Math.round(0x60 + ratio * (0x0f - 0x60));
                }} else {{
                    let ratio = (val - midE) / (maxE - midE);
                    r = Math.round(0xf1 + ratio * (0xe7 - 0xf1));
                    g = Math.round(0xc4 + ratio * (0x4c - 0xc4));
                    b = Math.round(0x0f + ratio * (0x3c - 0x0f));
                }}
                return `rgb(${{r}}, ${{g}}, ${{b}})`;
            }}

            function gambarRuteKePeta(arrayIndexHari) {{
                layerGroup.clearLayers(); 
                let bounds = []; 
                let markersDitaruh = new Set();
                
                arrayIndexHari.forEach(idx_hari => {{
                    const ruteSatuHari = dataRute[idx_hari];
                    ruteSatuHari.forEach(segmen => {{
                        const polyline = segmen.jalur;
                        for (let i = 0; i < polyline.length - 1; i++) {{
                            let p1 = polyline[i];
                            let p2 = polyline[i+1];
                            let elevRata = (p1[2] + p2[2]) / 2;
                            let line = L.polyline([[p1[0], p1[1]], [p2[0], p2[1]]], {{
                                color: getColorForElevation(elevRata),
                                weight: 5,
                                opacity: 0.85
                            }}).addTo(layerGroup);
                            bounds.push([p1[0], p1[1]]);
                        }}
                        
                        const titikAwal = polyline[0];
                        const titikAkhir = polyline[polyline.length - 1];
                        
                        if (!markersDitaruh.has(segmen.asal_nama)) {{
                            let warnaIkon = segmen.is_depot_asal ? 'orange' : '#3498db';
                            L.circleMarker([titikAwal[0], titikAwal[1]], {{
                                radius: segmen.is_depot_asal ? 8 : 6,
                                color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1
                            }}).bindPopup(`<b>${{segmen.asal_nama}}</b><br>Elevasi: ${{titikAwal[2].toFixed(1)}}m`).addTo(layerGroup);
                            markersDitaruh.add(segmen.asal_nama);
                        }}
                        
                        if (!markersDitaruh.has(segmen.tujuan_nama)) {{
                            let warnaIkon = segmen.is_depot_tujuan ? 'orange' : '#3498db';
                            L.circleMarker([titikAkhir[0], titikAkhir[1]], {{
                                radius: segmen.is_depot_tujuan ? 8 : 6,
                                color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1
                            }}).bindPopup(`<b>${{segmen.tujuan_nama}}</b><br>Elevasi: ${{titikAkhir[2].toFixed(1)}}m`).addTo(layerGroup);
                            markersDitaruh.add(segmen.tujuan_nama);
                        }}
                    }});
                }});
                
                if(bounds.length > 0) map.fitBounds(bounds, {{padding: [30, 30]}});
            }}

            function tampilkanSemuaRute() {{
                document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                let semuaIndex = dataMetrik.map((_, i) => i);
                gambarRuteKePeta(semuaIndex);
            }}

            const container = document.getElementById('cards-container');
            dataMetrik.forEach((metrik, idx) => {{
                let isSabtu = metrik.hari.toLowerCase().includes('sabtu');
                let badgeClass = isSabtu ? 'sabtu' : 'minggu';
                let badgeText = isSabtu ? 'HEAVY CLIMB' : 'RECOVERY';
                
                let card = document.createElement('div');
                card.className = 'card';
                card.innerHTML = `
                    <div class="badge ${{badgeClass}}">${{badgeText}}</div>
                    <div class="card-header">${{metrik.hari}}</div>
                    <div class="metric-row"><span>Total Jarak:</span> <span class="metric-val">${{metrik.jarak.toFixed(2)}} Km</span></div>
                    <div class="metric-row"><span>Fatigue Index:</span> <span class="metric-val">${{metrik.fatigue.toFixed(2)}}</span></div>
                    <div class="metric-row"><span>Skor Jalan:</span> <span class="metric-val">${{metrik.jalan.toFixed(2)}}</span></div>
                `;
                
                card.onclick = () => {{
                    document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                    card.classList.add('active'); 
                    gambarRuteKePeta([idx]);
                }};
                
                container.appendChild(card);
            }});

            window.onload = tampilkanSemuaRute;
        </script>
    </body>
    </html>
    """

    with open(OUTPUT_DASHBOARD, "w", encoding="utf-8") as file:
        file.write(html_content)

    print(f"🎉 SELESAI! Buka file ini di browsermu: {OUTPUT_DASHBOARD}")

# ─────────────────────────────────────────────────────────────────────
# 4. MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        matriks_jarak = pd.read_csv(os.path.join(TARGET_FOLDER, 'distance_matrix_osmnx.csv'), index_col=0)
        matriks_jalan = pd.read_csv(os.path.join(TARGET_FOLDER, 'road_condition_matrix_osmnx.csv'), index_col=0)
        matriks_elevasi = pd.read_csv(os.path.join(TARGET_FOLDER, 'elevation_matrix_surabaya.csv'), index_col=0)
        df_cluster = pd.read_csv(os.path.join(TARGET_FOLDER, 'destinasi_clustered_real_distance.csv'))
        
        ROUTE_JSON_PATH = os.path.join(TARGET_FOLDER, 'route_polyline_registry.json')
        with open(ROUTE_JSON_PATH, 'r') as f:
            route_registry = json.load(f)
            
        names = matriks_jarak.index.tolist()

        dist_np = matriks_jarak.to_numpy()
        road_np = matriks_jalan.to_numpy()
        elev_np = matriks_elevasi.to_numpy()
        cluster_np = df_cluster['Cluster_ID'].to_numpy()

        print("\n" + "=" * 65)
        print("🐜 TUNING PARAMETER ACO FINAL DENGAN DATASET OSMNX LOKAL")
        print("=" * 65)

        def objective_final(trial):
            alpha = trial.suggest_float('alpha', 0.1, 1.5)
            beta = trial.suggest_float('beta', 2.0, 8.0)
            evaporation = trial.suggest_float('evaporation', 0.05, 0.5)
            
            score, _, _ = run_aco_final(dist_np, elev_np, road_np, cluster_np, 
                                        alpha, beta, evaporation, max_km_user=MAX_KM_USER, num_ants=15, iterations=15)
            return score

        study = optuna.create_study(direction='minimize')
        study.optimize(objective_final, n_trials=20)
        best_p = study.best_params
        
        print(f"\n✅ Tuning Selesai! Skor Optuna Terbaik: {round(study.best_value, 2)}")
        print(f"✅ Parameter Terbaik: {best_p}")

        print(f"\n▶ Mengeksekusi Eksperimen Final ACO dengan Parameter Terbaik...")
        score_f, rute_f, log_f = run_aco_final(
            dist_np, elev_np, road_np, cluster_np,
            alpha=best_p['alpha'], beta=best_p['beta'], evaporation=best_p['evaporation'],
            max_km_user=MAX_KM_USER, num_ants=40, iterations=50
        )

        print("\n" + "=" * 65)
        print("====== HASIL OPTIMASI AKHIR MULTI-WEEKEND ITINERARY ACO ======")
        print("=" * 65)
        print(pd.DataFrame(log_f).to_string(index=False))
        print(f"\nSkor Akhir Fungsi Fitness: {round(score_f, 2)}")
        print(f"Total Waktu Liburan Keliling Surabaya: {len(rute_f)} Hari ({len(rute_f)//2} Akhir Pekan)")
        
        pd.DataFrame(log_f).to_csv(os.path.join(TARGET_FOLDER, 'itinerary_multi_weekend_aco.csv'), index=False)
        print(f"\nBerkas laporan skripsi berhasil diekspor ke folder: {TARGET_FOLDER}")
        
        generate_dashboard(rute_f, log_f, names, route_registry, TARGET_FOLDER)

    except FileNotFoundError as e:
        print(f"[ERROR] Berkas data preprocessing tidak ditemukan: {e}")
        print(f"Pastikan file csv & json sudah dipindahkan ke dalam folder: {TARGET_FOLDER}")

d:\Semester 6\SC\softcomputing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🐜 TUNING PARAMETER ACO FINAL DENGAN DATASET OSMNX LOKAL

✅ Tuning Selesai! Skor Optuna Terbaik: 23573.78
✅ Parameter Terbaik: {'alpha': 1.1484088804980648, 'beta': 3.966377522008151, 'evaporation': 0.4076022749863751}

▶ Mengeksekusi Eksperimen Final ACO dengan Parameter Terbaik...

====== HASIL OPTIMASI AKHIR MULTI-WEEKEND ITINERARY ACO ======
                    Hari/Trip  Jarak (Km)  Fatigue Index  Skor Jalan OSMnx  P_Fatigue  P_Cluster
Pekan 1 - Sabtu (Jalan Berat)       24.90         699.09             52.29    1048.63        0.0
  Pekan 1 - Minggu (Recovery)       24.43         444.05             44.85    3552.36      100.0
Pekan 2 - Sabtu (Jalan Berat)       24.18        4701.83             23.53    7052.75       50.0
  Pekan 2 - Minggu (Recovery)       20.18         576.80             16.58    4614.42        0.0
Pekan 3 - Sabtu (Jalan Berat)       20.54        3141.07              8.07    4711.61        0.0

Skor Akhir Fungsi Fitness: 23148.69
Total Waktu Liburan Keliling Sura

In [3]:
import os
import pandas as pd
import numpy as np
import random
import optuna
import json

# Sembunyikan log Optuna agar terminal komputasi bersih rapi
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─────────────────────────────────────────────────────────────────────
# PARAMETER GLOBAL & KONFIGURASI (Identik dengan Temenmu)
# ─────────────────────────────────────────────────────────────────────
TARGET_FOLDER = os.path.join('osmnx_inputs')
MAX_KM_USER = 25.0

# ─────────────────────────────────────────────────────────────────────
# 1. FUNGSI FITNESS GA (MULTI-WEEKEND & TERKALIBRASI LOKAL)
# ─────────────────────────────────────────────────────────────────────
def hitung_fitness_ga_final(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user):
    w_jarak = 10.0
    w_jalan = 5.0
    w_penalti_cluster = 50.0 
    w_hari = 30.0  
    
    fitness_total = len(solusi_multi_hari) * w_hari
    log_itinerary = []
    
    for idx_hari, rute in enumerate(solusi_multi_hari, start=1):
        if idx_hari % 2 != 0:
            nama_hari = f"Pekan {(idx_hari+1)//2} - Sabtu (Jalan Berat)"
            w_penalti_fatigue = 1.5; w_penalti_jarak = 5.0     
        else:
            nama_hari = f"Pekan {idx_hari//2} - Minggu (Recovery)"
            w_penalti_fatigue = 8.0; w_penalti_jarak = 25.0    
            
        jarak_m = fatigue = skor_jalan = p_cluster = 0.0
        
        for k in range(len(rute) - 1):
            a, t = rute[k], rute[k+1]
            
            dm = dist_np[a, t]
            dz = elev_np[a, t]
            js = road_np[a, t]
            
            jarak_m += dm
            skor_jalan += js
            
            if dz > 0 and dm > 0: 
                fatigue += dm * ((dz / dm) ** 2) * 400.0
            else: 
                fatigue += dm * 0.0001
                
            if a != 0 and t != 0:
                if cluster_np[a-1] != cluster_np[t-1]:
                    p_cluster += w_penalti_cluster

        jkm = jarak_m / 1000.0
        p_jarak = max(0.0, jkm - max_km_user) * w_penalti_jarak  
        p_fatigue = fatigue * w_penalti_fatigue
            
        fitness_hari = (jkm * w_jarak) + (skor_jalan * w_jalan) + p_jarak + p_fatigue + p_cluster
        fitness_total += fitness_hari
        
        log_itinerary.append({
            "Hari/Trip": nama_hari, "Jarak (Km)": round(jkm, 2),
            "Fatigue Index": round(fatigue, 2), "Skor Jalan OSMnx": round(skor_jalan, 2),
            "P_Fatigue": round(p_fatigue, 2), "P_Cluster": round(p_cluster, 2)
        })
        
    return fitness_total, log_itinerary

# ─────────────────────────────────────────────────────────────────────
# 2. MECHANISM GA ENGINE (Kromosom, Crossover, Mutation, Selection)
# ─────────────────────────────────────────────────────────────────────
def buat_kromosom_acak(n):
    k = list(range(1, n+1)); random.shuffle(k); return k

def decode_kromosom(kromo, dist_np, max_km_user):
    max_m = max_km_user * 1000; hasil = []; sisa = list(kromo)
    while sisa:
        rute = [0]; jarak = 0.0
        while sisa:
            pos = rute[-1]; tuj = sisa[0]
            dtuj = dist_np[pos, tuj]
            # Logika potong hari lunak penyesuaian multi-hari
            if (jarak + dtuj > max_m) and (random.random() > 0.15): break
            rute.append(tuj); jarak += dtuj; sisa.pop(0)
        rute.append(0); hasil.append(rute)
    return hasil

def order_crossover(p1, p2):
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    anak = [None]*n; anak[a:b+1] = p1[a:b+1]
    sisa = [g for g in p2 if g not in anak[a:b+1]]
    ptr = 0
    for i in range(n):
        if anak[i] is None: anak[i] = sisa[ptr]; ptr += 1
    return anak

def swap_mutation(k, prob):
    k = k[:]
    if random.random() < prob:
        i, j = random.sample(range(len(k)), 2); k[i], k[j] = k[j], k[i]
    return k

def tournament_selection(pop, fit, k):
    idx = random.sample(range(len(pop)), k)
    return pop[min(idx, key=lambda i: fit[i])][:]

# ─────────────────────────────────────────────────────────────────────
# 3. CORE ENGINE GA RUNNER (Identik Alur Pemanggilan ACO Temenmu)
# ─────────────────────────────────────────────────────────────────────
def run_ga_final(dist_np, elev_np, road_np, cluster_np, pop_size, prob_cross, prob_mut, tournament_k, max_km_user, iterations=100):
    n_dest = len(dist_np) - 1
    populasi = [buat_kromosom_acak(n_dest) for _ in range(pop_size)]
    
    best_score = float('inf')
    best_route = []
    best_log = None
    
    for gen in range(1, iterations + 1):
        fit_list = []; log_list = []
        for k in populasi:
            rute = decode_kromosom(k, dist_np, max_km_user)
            f_score, lg = hitung_fitness_ga_final(rute, dist_np, elev_np, road_np, cluster_np, max_km_user)
            fit_list.append(f_score); log_list.append(lg)

        idx_best = int(np.argmin(fit_list))
        if fit_list[idx_best] < best_score:
            best_score = fit_list[idx_best]
            best_route = decode_kromosom(populasi[idx_best], dist_np, max_km_user)
            best_log = log_list[idx_best]

        # Evolusi Reproduksi Populasi Baru
        sorted_idx = np.argsort(fit_list)
        elit = [populasi[i][:] for i in sorted_idx[:2]] # N_ELITE = 2
        generasi_baru = elit[:]
        while len(generasi_baru) < pop_size:
            p1 = tournament_selection(populasi, fit_list, tournament_k)
            p2 = tournament_selection(populasi, fit_list, tournament_k)
            if random.random() < prob_cross:
                a1 = order_crossover(p1, p2); a2 = order_crossover(p2, p1)
            else:
                a1, a2 = p1[:], p2[:]
            generasi_baru.append(swap_mutation(a1, prob_mut))
            if len(generasi_baru) < pop_size: generasi_baru.append(swap_mutation(a2, prob_mut))
        populasi = generasi_baru
        
    return best_score, best_route, best_log

# ─────────────────────────────────────────────────────────────────────
# 4. GENERATE DASHBOARD HTML (Membaca Registri Polylines Lokal)
# ─────────────────────────────────────────────────────────────────────
def generate_dashboard(rute_terbaik, log_metrik, names, route_registry, target_folder):
    print("\n" + "=" * 65)
    print("🌍 MEMBUAT DASHBOARD PETA INTERAKTIF KEDUA (GA LAYERS)...")
    
    OUTPUT_DASHBOARD = os.path.join(target_folder, 'dashboard_interaktif_ga.html')
    
    data_metrik = []
    for log in log_metrik:
        data_metrik.append({
            "hari": log["Hari/Trip"], "jarak": log["Jarak (Km)"],
            "fatigue": log["Fatigue Index"], "jalan": log["Skor Jalan OSMnx"]
        })
        
    data_rute_js = []
    for rute in rute_terbaik:
        hari_data = []
        for k in range(len(rute) - 1):
            idx_asal = rute[k]; idx_tujuan = rute[k+1]
            key = f"{idx_asal}_{idx_tujuan}"
            
            if key in route_registry:
                hari_data.append({
                    "asal_nama": names[idx_asal], "tujuan_nama": names[idx_tujuan],
                    "is_depot_asal": (idx_asal == 0), "is_depot_tujuan": (idx_tujuan == 0),
                    "jalur": route_registry[key] 
                })
        data_rute_js.append(hari_data)

    json_rute_str = json.dumps(data_rute_js)
    json_metrik_str = json.dumps(data_metrik)

    # Memuat template HTML Interaktif Leaflet milik temenmu agar seragam
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Dashboard Optimasi Rute Sepeda GA - Surabaya</title>
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
        <style>
            body {{ margin: 0; padding: 0; font-family: 'Inter', sans-serif; display: flex; height: 100vh; background-color: #f8f9fa; }}
            #map-container {{ flex: 1; height: 100%; position: relative; }}
            #map {{ height: 100%; width: 100%; }}
            .legend {{ position: absolute; bottom: 30px; left: 30px; z-index: 1000; background: white; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
            .legend-title {{ font-weight: bold; margin-bottom: 8px; font-size: 14px; }}
            .gradient-bar {{ width: 250px; height: 15px; border-radius: 4px; background: linear-gradient(to right, #27ae60, #f1c40f, #e74c3c); }}
            .legend-labels {{ display: flex; justify-content: space-between; margin-top: 5px; font-size: 12px; color: #555; }}
            #sidebar {{ width: 400px; height: 100%; background: white; box-shadow: -2px 0 10px rgba(0,0,0,0.1); overflow-y: auto; padding: 20px; box-sizing: border-box; z-index: 1000; }}
            h2 {{ margin-top: 0; color: #2c3e50; font-size: 20px; border-bottom: 2px solid #2980b9; padding-bottom: 10px; }}
            .card {{ background: #ffffff; border: 1px solid #e0e6ed; border-radius: 8px; padding: 15px; margin-bottom: 15px; cursor: pointer; transition: all 0.3s ease; }}
            .card:hover {{ transform: translateY(-3px); box-shadow: 0 5px 15px rgba(0,0,0,0.08); border-color: #2980b9; }}
            .card.active {{ border: 2px solid #2980b9; background-color: #f0f7fb; }}
            .card-header {{ font-weight: 700; color: #2c3e50; margin-bottom: 10px; font-size: 15px; }}
            .badge {{ display: inline-block; padding: 3px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; color: white; margin-bottom: 10px; }}
            .badge.sabtu {{ background-color: #e74c3c; }}
            .badge.minggu {{ background-color: #27ae60; }}
            .metric-row {{ display: flex; justify-content: space-between; margin-bottom: 5px; font-size: 13px; color: #555; }}
            .metric-val {{ font-weight: bold; color: #2c3e50; }}
            #btn-semua {{ width: 100%; padding: 12px; background: #2c3e50; color: white; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; margin-bottom: 20px; transition: background 0.2s; }}
        </style>
    </head>
    <body>
        <div id="map-container">
            <div id="map"></div>
            <div class="legend">
                <div class="legend-title">Elevasi Tanjakan Rute GA (m)</div>
                <div class="gradient-bar"></div>
                <div class="legend-labels"><span>0m (Datar)</span><span>Landai</span><span>>25m (Berat)</span></div>
            </div>
        </div>
        <div id="sidebar">
            <h2>Itinerary Multi-Hari GA</h2>
            <button id="btn-semua" onclick="tampilkanSemuaRute()">Tampilkan Seluruh Rute</button>
            <div id="cards-container"></div>
        </div>
        <script>
            const dataRute = {json_rute_str};
            const dataMetrik = {json_metrik_str};
            
            const map = L.map('map').setView([-7.262015, 112.739727], 13);
            L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
                attribution: '&copy; OpenStreetMap contributors &copy; CARTO'
            }}).addTo(map);

            let layerGroup = L.layerGroup().addTo(map);

            function getColorForElevation(elev) {{
                const minE = 0, midE = 12.5, maxE = 25;
                let val = Math.max(minE, Math.min(elev, maxE)); 
                let r, g, b;
                if (val <= midE) {{
                    let ratio = val / midE;
                    r = Math.round(0x27 + ratio * (0xf1 - 0x27)); g = Math.round(0xae + ratio * (0xc4 - 0xae)); b = Math.round(0x60 + ratio * (0x0f - 0x60));
                }} else {{
                    let ratio = (val - midE) / (maxE - midE);
                    r = Math.round(0xf1 + ratio * (0xe7 - 0xf1)); g = Math.round(0xc4 + ratio * (0x4c - 0xc4)); b = Math.round(0x0f + ratio * (0x3c - 0x0f));
                }}
                return `rgb(${{r}}, ${{g}}, ${{b}})`;
            }}

            function gambarRuteKePeta(arrayIndexHari) {{
                layerGroup.clearLayers(); let bounds = []; let markersDitaruh = new Set();
                arrayIndexHari.forEach(idx_hari => {{
                    const ruteSatuHari = dataRute[idx_hari];
                    ruteSatuHari.forEach(segmen => {{
                        const polyline = segmen.jalur;
                        for (let i = 0; i < polyline.length - 1; i++) {{
                            let p1 = polyline[i]; let p2 = polyline[i+1]; let elevRata = (p1[2] + p2[2]) / 2;
                            L.polyline([[p1[0], p1[1]], [p2[0], p2[1]]], {{ color: getColorForElevation(elevRata), weight: 5, opacity: 0.85 }}).addTo(layerGroup);
                            bounds.push([p1[0], p1[1]]);
                        }}
                        const titikAwal = polyline[0]; const titikAkhir = polyline[polyline.length - 1];
                        if (!markersDitaruh.has(segmen.asal_nama)) {{
                            let warnaIkon = segmen.is_depot_asal ? 'orange' : '#3498db';
                            L.circleMarker([titikAwal[0], titikAwal[1]], {{ radius: segmen.is_depot_asal ? 8 : 6, color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1 }}).bindPopup(`<b>${{segmen.asal_nama}}</b>`).addTo(layerGroup);
                            markersDitaruh.add(segmen.asal_nama);
                        }}
                        if (!markersDitaruh.has(segmen.tujuan_nama)) {{
                            let warnaIkon = segmen.is_depot_tujuan ? 'orange' : '#3498db';
                            L.circleMarker([titikAkhir[0], titikAkhir[1]], {{ radius: segmen.is_depot_tujuan ? 8 : 6, color: 'white', weight: 2, fillColor: warnaIkon, fillOpacity: 1 }}).bindPopup(`<b>${{segmen.tujuan_nama}}</b>`).addTo(layerGroup);
                            markersDitaruh.add(segmen.tujuan_nama);
                        }}
                    }});
                }});
                if(bounds.length > 0) map.fitBounds(bounds, {{padding: [30, 30]}});
            }}

            function tampilkanSemuaRute() {{
                document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                gambarRuteKePeta(dataMetrik.map((_, i) => i));
            }}

            const container = document.getElementById('cards-container');
            dataMetrik.forEach((metrik, idx) => {{
                let isSabtu = metrik.hari.toLowerCase().includes('sabtu');
                let badgeClass = isSabtu ? 'sabtu' : 'minggu';
                let badgeText = isSabtu ? 'HEAVY CLIMB' : 'RECOVERY';
                let card = document.createElement('div');
                card.className = 'card';
                card.innerHTML = `
                    <div class="badge ${{badgeClass}}">${{badgeText}}</div>
                    <div class="card-header">${{metrik.hari}}</div>
                    <div class="metric-row"><span>Total Jarak:</span> <span class="metric-val">${{metrik.jarak.toFixed(2)}} Km</span></div>
                    <div class="metric-row"><span>Fatigue Index:</span> <span class="metric-val">${{metrik.fatigue.toFixed(2)}}</span></div>
                    <div class="metric-row"><span>Skor Jalan:</span> <span class="metric-val">${{metrik.jalan.toFixed(2)}}</span></div>
                `;
                card.onclick = () => {{
                    document.querySelectorAll('.card').forEach(c => c.classList.remove('active'));
                    card.classList.add('active'); gambarRuteKePeta([idx]);
                }};
                container.appendChild(card);
            }});
            window.onload = tampilkanSemuaRute;
        </script>
    </body>
    </html>
    """
    with open(OUTPUT_DASHBOARD, "w", encoding="utf-8") as file:
        file.write(html_content)
    print(f"🎉 DASHBOARD GA SELESAI! Silakan buka file: {OUTPUT_DASHBOARD}")

# ─────────────────────────────────────────────────────────────────────
# 5. MAIN EXECUTION (100% Identik Alur dengan Milik Temenmu)
# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        # Load database lokal dari folder 'osmnx_inputs' seperti kode temenmu
        matriks_jarak = pd.read_csv(os.path.join(TARGET_FOLDER, 'distance_matrix_osmnx.csv'), index_col=0)
        matriks_jalan = pd.read_csv(os.path.join(TARGET_FOLDER, 'road_condition_matrix_osmnx.csv'), index_col=0)
        matriks_elevasi = pd.read_csv(os.path.join(TARGET_FOLDER, 'elevation_matrix_surabaya.csv'), index_col=0)
        df_cluster = pd.read_csv(os.path.join(TARGET_FOLDER, 'destinasi_clustered_real_distance.csv'))
        
        ROUTE_JSON_PATH = os.path.join(TARGET_FOLDER, 'route_polyline_registry.json')
        with open(ROUTE_JSON_PATH, 'r') as f:
            route_registry = json.load(f)
            
        names = matriks_jarak.index.tolist()

        dist_np = matriks_jarak.to_numpy()
        road_np = matriks_jalan.to_numpy()
        elev_np = matriks_elevasi.to_numpy()
        cluster_np = df_cluster['Cluster_ID'].to_numpy()

        print("\n" + "="*65)
        print("🤖 [FASE 1] RUNNING OPTUNA HYPERPARAMETER TUNING GA ENGINE 🤖")
        print("="*65)

        # Definisikan objektif tuning parameter untuk GA dengan pencarian lebih luas
        def objective_final(trial):
            # Memaksa Optuna memilih ukuran populasi yang lebih besar agar variasi rute kaya
            pop_size = trial.suggest_int('pop_size', 80, 160, step=20) 
            prob_cross = trial.suggest_float('prob_crossover', 0.75, 0.95)
            prob_mut = trial.suggest_float('prob_mutasi', 0.05, 0.20)
            tournament_k = trial.suggest_int('tournament_k', 4, 8)
            
            # Dinaikkan ke 30 generasi (sebelumnya 15) agar Optuna tidak salah menilai parameter bagus
            score, _, _ = run_ga_final(dist_np, elev_np, road_np, cluster_np, 
                                       pop_size, prob_cross, prob_mut, tournament_k, 
                                       max_km_user=MAX_KM_USER, iterations=30) 
            return score

        # Eksekusi Optuna Tuning dengan total percobaan dinaikkan ke 30 trials (sebelumnya 20)
        study = optuna.create_study(direction='minimize')
        study.optimize(objective_final, n_trials=30)
        best_p = study.best_params
        
        print(f"\n✅ Tuning Selesai! Skor Optuna GA Terbaik: {round(study.best_value, 2)}")
        print(f"✅ Parameter GA Terbaik Terkunci: {best_p}")

        # ─── EKSEKUSI MENU UTAMA (10 RUNS BERBASIS PARAMETER OPTUNA) ───
        print(f"\n▶️ [FASE 2] MEMULAI EVALUASI 10 RUNS FINAL BERBASIS PARAMETER OPTUNA...")
        print("=================================================================")
        
        TOTAL_RUNS = 10
        history_akhir_runs = []
        score_f = float('inf')
        rute_f, log_f = None, None
        run_terbaik_id = 0

        for id_run in range(1, TOTAL_RUNS + 1):
            # Jalankan penuh 100 generasi menggunakan racikan angka mutlak dari Optuna
            score_curr, rute_curr, log_curr = run_ga_final(
                dist_np, elev_np, road_np, cluster_np,
                pop_size=int(best_p['pop_size']), prob_cross=best_p['prob_crossover'], 
                prob_mut=best_p['prob_mutasi'], tournament_k=int(best_p['tournament_k']),
                max_km_user=MAX_KM_USER, iterations=100
            )
            history_akhir_runs.append(score_curr)
            print(f"   - [RUN {id_run:02d}/10] Selesai | Fitness Akhir: {score_curr:,.2f} | Trip: {len(rute_curr)} Hari")
            
            if score_curr < score_f:
                score_f = score_curr; rute_f = rute_curr; log_f = log_curr; run_terbaik_id = id_run

        print("\n" + "=" * 65)
        print("====== HASIL OPTIMASI AKHIR MULTI-WEEKEND ITINERARY GA ======")
        print("=" * 65)
        print(pd.DataFrame(log_f).to_string(index=False))
        print(f"\nSkor Akhir Fungsi Fitness GA Global : {round(score_f, 2)}")
        print(f"Total Percobaan Terbaik Ditemukan Pada : [RUN {run_terbaik_id}]")
        print(f"Total Waktu Liburan Keliling Surabaya : {len(rute_f)} Hari")
        
        # Ekspor berkas CSV
        pd.DataFrame(log_f).to_csv(os.path.join(TARGET_FOLDER, 'itinerary_multi_weekend_ga.csv'), index=False)
        print(f"\nBerkas laporan CSV berhasil diekspor ke folder: {TARGET_FOLDER}")
        
        # Gambar Dashboard Leaflet Melik-Liuk dari JSON lokal
        generate_dashboard(rute_f, log_f, names, route_registry, TARGET_FOLDER)

    except FileNotFoundError as e:
        print(f"[ERROR] Berkas data preprocessing tidak ditemukan: {e}")
        print(f"Pastikan file csv & json sudah berada di dalam folder: {TARGET_FOLDER}")


🤖 [FASE 1] RUNNING OPTUNA HYPERPARAMETER TUNING GA ENGINE 🤖

✅ Tuning Selesai! Skor Optuna GA Terbaik: 36651.47
✅ Parameter GA Terbaik Terkunci: {'pop_size': 120, 'prob_crossover': 0.8828702292234409, 'prob_mutasi': 0.09797365504610914, 'tournament_k': 7}

▶️ [FASE 2] MEMULAI EVALUASI 10 RUNS FINAL BERBASIS PARAMETER OPTUNA...
   - [RUN 01/10] Selesai | Fitness Akhir: 39,472.07 | Trip: 11 Hari
   - [RUN 02/10] Selesai | Fitness Akhir: 39,878.84 | Trip: 11 Hari
   - [RUN 03/10] Selesai | Fitness Akhir: 34,734.38 | Trip: 10 Hari
   - [RUN 04/10] Selesai | Fitness Akhir: 31,453.00 | Trip: 10 Hari
   - [RUN 05/10] Selesai | Fitness Akhir: 37,428.92 | Trip: 10 Hari
   - [RUN 06/10] Selesai | Fitness Akhir: 36,691.62 | Trip: 10 Hari
   - [RUN 07/10] Selesai | Fitness Akhir: 33,021.48 | Trip: 9 Hari
   - [RUN 08/10] Selesai | Fitness Akhir: 33,579.87 | Trip: 9 Hari
   - [RUN 09/10] Selesai | Fitness Akhir: 33,497.07 | Trip: 9 Hari
   - [RUN 10/10] Selesai | Fitness Akhir: 27,297.07 | Trip: 7